# Stage 2 Notebook 67 - Exp2LLL Anchor + cls_sep + IoU-priority matching (tempered) + 14ep

**Re-run NB66 with tempered IoU priority + longer training.** NB66 (exp61) failed mid-run with `tar: Transport endpoint is not connected` — Drive mount dropped. This is a fresh retry with slightly less aggressive IoU priority and longer training.

NB62 used `cost_iou=2.0, cost_point=5.0, w_iou=2.0`. NB66 (failed) tried `cost_iou=5.0, cost_point=2.0, w_iou=3.0` -- full reversal. Exp2LLL: midpoint at `cost_iou=4.0, cost_point=3.0, w_iou=2.5`. Plus 14 epochs (vs NB62's 12). Tests whether moderate IoU-priority matching helps while preserving NB62's coordinate-distance signal.

Diffs vs NB62 (exp57):
- `match_cost_iou: 2.0 -> 4.0`
- `match_cost_point: 5.0 -> 3.0`
- `w_iou: 2.0 -> 2.5`
- `end_epoch: 12 -> 14`

### Run mode
1. Smoke.
2. 14 epochs full 70K. ~3-3.5 hr.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp62_rmt_gca_anchor_cls_sep_vfl_iou_match_long14_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp62_rmt_gca_anchor_cls_sep_vfl_iou_match_long14_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp62_rmt_gca_anchor_cls_sep_vfl_iou_match_long14_joint_smoke.log
OK exp62_rmt_gca_anchor_cls_sep_vfl_iou_match_long14_joint.yaml
  lane_shape=(1, 16, 72, 2) det_shape=(1, 4, 4)
  lane_loss=5.5912 det_loss=3.4628 grad_cos=0.0520 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.49809741973876953, 'gate/lane_mean': 0.4975186288356781, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp62_rmt_gca_anchor_cls_sep_vfl_iou_match_long14_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'full14'
    EPOCHS = 14
    BATCH_SIZE = 8
    LIMIT_TRAIN = None
    LIMIT_VAL = 2000
    PRINT_EVERY = 50

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]
if LIMIT_TRAIN is not None:
    cmd.extend(['--limit-train', str(LIMIT_TRAIN)])

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('LIMIT_TRAIN:', LIMIT_TRAIN, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
LIMIT_TRAIN: None
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp62_rmt_gca_anchor_cls_sep_vfl_iou_match_long14_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp62_rmt_gca_anchor_cls_sep_vfl_iou_match_long14_joint_full14 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp62_rmt_gca_anchor_cls_sep_vfl_iou_match_long14_joint_full14.tar --epochs 14 --batch-size 8 --limit-val 2000 --force-extract --print-every 50
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp62_rmt_gca_anchor_cls_sep_vfl_iou_match_long14_joint_full14.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp62_rmt_gca_anchor_cls_sep_vfl_iou_match_long14_joint_full14_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp62_rmt

0

## What to watch in Exp2LLL

Reference NB62 (cost_iou=2, cost_point=5): matched_iou=0.550, decoded_f1=0.073, gap=0.045.

Pass criteria at epoch 14:
- val/matched_line_iou >= 0.55 (preserved or improved by tempered IoU-priority).
- val/lane/decoded_f1 >= 0.08 (beat NB62 by 10 %).
- pos-neg gap >= 0.05.
- val/lane_f1 >= 0.13.